# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


## Objective

This notebook implements the **Dense Retrieval** model as the second baseline for evaluating the proposed **Root-Aware Lexical Semantic Retrieval (RALSR)** framework.

Unlike the BM25 baseline, which relies exclusively on lexical keyword matching, Dense Retrieval represents both questions and Qur'anic passages as continuous semantic vector embeddings generated by a pre-trained transformer model. Retrieval is performed by measuring the similarity between the query embedding and the passage embeddings, allowing semantically related passages to be retrieved even when they do not share identical lexical terms.

The primary objective of this notebook is to establish a strong semantic retrieval baseline that complements the lexical BM25 baseline and provides a comprehensive comparison with the proposed RALSR framework.

The retrieval results produced in this notebook will later be compared with:

- **Baseline 1:** BM25 Sparse Retrieval (Notebook 10A)
- **Baseline 2:** Dense Retrieval (this notebook)
- **Proposed Method A:** Root-Aware Lexical Semantic Retrieval (RALSR)
- **Proposed Method B:** RALSR + Transformer Cross-Encoder Re-ranking (Notebook 11)

The ranked retrieval results generated in this notebook will subsequently be evaluated in **Notebook 10C** using standard Information Retrieval evaluation metrics, including Mean Reciprocal Rank (MRR), Recall@K, Mean Average Precision (MAP), and Normalized Discounted Cumulative Gain (NDCG).

---

# Workflow

```text
Preprocessed Questions
           │
           ▼
Processed Qur'anic Passage Collection
           │
           ▼
Transformer Embedding Model
           │
           ▼
Query Embedding Generation
           │
           ▼
Passage Embedding Generation
           │
           ▼
Cosine Similarity Computation
           │
           ▼
Passage Ranking
           │
           ▼
Top-100 Dense Retrieval
           │
           ▼
Export Ranked Retrieval Results
           │
           ▼
Comparative Evaluation (Notebook 10C)
```

---

# Input Datasets

## 1. Preprocessed Query Collection

Generated in **Notebook 4 – Arabic Query Preprocessing**

**File**

`QuranQA_Preprocessed_Queries.csv`

Contains:

- Question_ID
- Question
- Query_Terms

---

## 2. Processed Qur'anic Passage Collection

Generated in **Notebook 8 – Preparing the Quran Passage Collection for Retrieval**

**File**

`QuranQA_Processed_Passage_Collection.csv`

Contains:

- Passage_ID
- Passage_Text

---

# Dense Retrieval Pipeline

The Dense Retrieval baseline follows the standard semantic retrieval paradigm:

1. Load the preprocessed questions.
2. Load the processed Qur'anic passage collection.
3. Generate transformer embeddings for all passages.
4. Generate transformer embeddings for each query.
5. Compute cosine similarity between query and passage embeddings.
6. Rank passages according to similarity scores.
7. Retain the Top-100 retrieved passages for each question.
8. Export the ranked retrieval results.

Unlike the proposed RALSR framework, Dense Retrieval does **not** perform:

- Hybrid root extraction
- Canonical Lexical Mapping (CLM)
- Semantic enrichment using *Maqāyīs al-Lughah*
- Linguistic Evidence Model (LEM)
- Knowledge-driven feature engineering

Instead, semantic similarity is learned implicitly through transformer-generated vector representations.

---

# Output

This notebook produces:

`QuranQA_Dense_Retrieval.csv`

The exported dataset contains:

- Question_ID
- Question
- Query_Terms
- Passage_ID
- Dense_Score
- Rank

The resulting retrieval list serves as the **semantic retrieval baseline** for the comparative evaluation performed in **Notebook 10C**.

In [12]:
# ============================================================
# Import Required Libraries
# Notebook 10B – Baseline 2: Dense Retrieval
# ============================================================

import ast
from pathlib import Path

import numpy as np
import pandas as pd

import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# Project Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parents[2]

CODE_DIR = PROJECT_ROOT / "04_Code"

MODELS_DIR = CODE_DIR / "Models"

MODEL_PATH = MODELS_DIR / "multilingual-e5-base"

EXPERIMENTS_DIR = PROJECT_ROOT / "05_Experiments"

PREPROCESSING_DIR = (
    EXPERIMENTS_DIR /
    "Preprocessing"
)

PASSAGE_PREPARATION_DIR = (
    EXPERIMENTS_DIR /
    "Passage_Preparation"
)

DENSE_RETRIEVAL_DIR = (
    EXPERIMENTS_DIR /
    "Dense_Retrieval"
)

# Create output directory if it does not already exist
DENSE_RETRIEVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Device Configuration
# ------------------------------------------------------------

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("PROJECT PATHS INITIALIZED")
print("=" * 70)

print(f"Project Root       : {PROJECT_ROOT}")
print(f"Models Folder      : {MODELS_DIR}")
print(f"Embedding Model    : {MODEL_PATH.name}")
print(f"Experiments Folder : {EXPERIMENTS_DIR}")
print(f"Output Folder      : {DENSE_RETRIEVAL_DIR}")
print(f"Processing Device  : {DEVICE}")

PROJECT PATHS INITIALIZED
Project Root       : ${LOCAL_USER_HOME}\Desktop\Alatoo university CS\NLP Thesis
Models Folder      : ${THESIS_MODEL_ROOT}
Embedding Model    : multilingual-e5-base
Experiments Folder : ${LOCAL_USER_HOME}\Desktop\Alatoo university CS\NLP Thesis\05_Experiments
Output Folder      : ${LOCAL_USER_HOME}\Desktop\Alatoo university CS\NLP Thesis\05_Experiments\Dense_Retrieval
Processing Device  : cpu


## Load Dense Retrieval Resources

The preprocessed QuranQA query collection and the processed Qur'anic passage collection are loaded into memory.

To ensure reproducibility and prevent downstream errors, the input datasets are validated by verifying the presence of the required columns before proceeding with the embedding generation stage.

The loaded datasets constitute the input to the Dense Retrieval pipeline implemented in this notebook.

In [13]:
# ============================================================
# Load Query and Passage Collections
# ============================================================

import ast

# ------------------------------------------------------------
# Input Files
# ------------------------------------------------------------

QUERY_FILE = (
    PREPROCESSING_DIR /
    "QuranQA_Preprocessed_Queries.csv"
)

PASSAGE_FILE = (
    PASSAGE_PREPARATION_DIR /
    "QuranQA_Processed_Passage_Collection.csv"
)

# ------------------------------------------------------------
# Load Datasets
# ------------------------------------------------------------

queries_df = pd.read_csv(QUERY_FILE)

passages_df = pd.read_csv(PASSAGE_FILE)

# ------------------------------------------------------------
# Convert Query Terms to Python Lists
# ------------------------------------------------------------

queries_df["Query_Terms"] = queries_df["Query_Terms"].apply(ast.literal_eval)

# ------------------------------------------------------------
# Validate Required Columns
# ------------------------------------------------------------

required_query_columns = {
    "Question_ID",
    "Question",
    "Query_Terms"
}

required_passage_columns = {
    "Passage_ID",
    "Passage_Text"
}

missing_query = required_query_columns - set(queries_df.columns)
missing_passage = required_passage_columns - set(passages_df.columns)

if missing_query:
    raise ValueError(
        f"Missing query columns: {sorted(missing_query)}"
    )

if missing_passage:
    raise ValueError(
        f"Missing passage columns: {sorted(missing_passage)}"
    )

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("=" * 70)
print("DENSE RETRIEVAL RESOURCES LOADED")
print("=" * 70)

print(f"Questions : {len(queries_df):,}")
print(f"Passages  : {len(passages_df):,}")

print("\nQuery Dataset Columns")

print(list(queries_df.columns))

print("\nPassage Dataset Columns")

print(list(passages_df.columns))

print("\nSample Query")

display(
    queries_df[
        [
            "Question_ID",
            "Question",
            "Query_Terms"
        ]
    ].head(1)
)

print("\nSample Passage")

display(
    passages_df[
        [
            "Passage_ID",
            "Passage_Text"
        ]
    ].head(1)
)

DENSE RETRIEVAL RESOURCES LOADED
Questions : 251
Passages  : 1,266

Query Dataset Columns
['Dataset', 'Question_ID', 'Question', 'Normalized_Question', 'Clean_Question', 'Tokens', 'Content_Tokens', 'Query_Terms']

Passage Dataset Columns
['Passage_ID', 'Passage_Text', 'Surah', 'Start_Ayah', 'End_Ayah', 'Ayah_Count', 'Character_Count', 'Word_Count']

Sample Query


,Question_ID,Question,Query_Terms
0,101,من هم قوم شعيب؟,"[قوم, شعيب]"



Sample Passage


,Passage_ID,Passage_Text
0,1:1-4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين....


## Load the Dense Retrieval Model

The **`intfloat/multilingual-e5-base`** transformer model is loaded to generate dense semantic representations for both questions and Qur'anic passages.

Unlike BM25, which relies on lexical term matching, this model converts textual inputs into continuous vector embeddings. Semantic retrieval is then performed by measuring the cosine similarity between the query and passage embeddings.

To verify that the model has been loaded correctly, a sample query is encoded and the dimensionality of the resulting embedding vector is displayed.

In [14]:
# ============================================================
# Load Local multilingual-e5-base Embedding Model
# ============================================================

print("=" * 70)
print("LOADING DENSE RETRIEVAL MODEL")
print("=" * 70)

print("Embedding Model : multilingual-e5-base (Local)")
print(f"Model Path      : {MODEL_PATH}")
print(f"Device          : {DEVICE}")

# ------------------------------------------------------------
# Load Sentence Transformer Model
# ------------------------------------------------------------

model = SentenceTransformer(
    str(MODEL_PATH),
    device=DEVICE
)

print("\nModel loaded successfully.")

# ------------------------------------------------------------
# Verify the Model
# ------------------------------------------------------------

test_query = "query: من هم قوم شعيب؟"

test_embedding = model.encode(
    test_query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nModel Verification")
print("-" * 40)

print(f"Test Query         : {test_query}")
print(f"Embedding Shape    : {test_embedding.shape}")
print(f"Embedding Dimension: {len(test_embedding)}")
print(f"Vector Norm        : {np.linalg.norm(test_embedding):.4f}")

print("\nFirst 10 Dimensions")

print(np.round(test_embedding[:10], 4))

LOADING DENSE RETRIEVAL MODEL
Embedding Model : multilingual-e5-base (Local)
Model Path      : ${THESIS_MODEL_ROOT}\multilingual-e5-base
Device          : cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2631.02it/s]



Model loaded successfully.

Model Verification
----------------------------------------
Test Query         : query: من هم قوم شعيب؟
Embedding Shape    : (768,)
Embedding Dimension: 768
Vector Norm        : 1.0000

First 10 Dimensions
[ 0.0221  0.0436 -0.0191  0.0262  0.0534 -0.001  -0.024  -0.0404  0.0222
 -0.0311]


## Generate and Cache Passage Embeddings

This step converts every Qur'anic passage into a dense semantic vector using the pre-trained **multilingual-e5-base** embedding model.

Each passage is prefixed with `"passage: "` following the recommended input format for the E5 model. The resulting embeddings are normalized to unit length and stored for efficient reuse in subsequent experiments.

To avoid unnecessary computation, the notebook first checks whether previously generated embeddings already exist. If available, they are loaded directly from disk; otherwise, new embeddings are generated and saved.

The following artifacts are created:

- **Passage_Embeddings.npy** — Dense vector representations of all Qur'anic passages.
- **Passage_Metadata.csv** — Mapping between each embedding and its corresponding Qur'anic passage.

In [15]:
# ============================================================
# Generate and Cache Passage Embeddings
# ============================================================

import os

EMBEDDINGS_FILE = (
    DENSE_RETRIEVAL_DIR /
    "Passage_Embeddings.npy"
)

METADATA_FILE = (
    DENSE_RETRIEVAL_DIR /
    "Passage_Metadata.csv"
)

# ------------------------------------------------------------
# Load Existing Embeddings (if available)
# ------------------------------------------------------------

if EMBEDDINGS_FILE.exists():

    print("=" * 70)
    print("LOADING EXISTING PASSAGE EMBEDDINGS")
    print("=" * 70)

    passage_embeddings = np.load(EMBEDDINGS_FILE)

    passage_metadata = pd.read_csv(METADATA_FILE)

# ------------------------------------------------------------
# Otherwise Generate Embeddings
# ------------------------------------------------------------

else:

    print("=" * 70)
    print("GENERATING PASSAGE EMBEDDINGS")
    print("=" * 70)

    # Prepare E5 Input Format

    passage_texts = [
        "passage: " + text
        for text in passages_df["Passage_Text"]
    ]

    # Generate Embeddings

    passage_embeddings = model.encode(
        passage_texts,
        batch_size=16,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Save Embeddings

    np.save(
        EMBEDDINGS_FILE,
        passage_embeddings
    )

    # Save Metadata

    passage_metadata = passages_df[
        [
            "Passage_ID",
            "Passage_Text"
        ]
    ].copy()

    passage_metadata.to_csv(
        METADATA_FILE,
        index=False,
        encoding="utf-8-sig"
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nEmbedding Summary")
print("-" * 40)

print(f"Passages           : {len(passage_metadata):,}")
print(f"Embedding Shape    : {passage_embeddings.shape}")
print(f"Embedding Dimension: {passage_embeddings.shape[1]}")

print("\nSample Passage")

display(
    passage_metadata.head(1)
)

print("\nFirst 10 Values of First Embedding")

print(
    np.round(
        passage_embeddings[0][:10],
        4
    )
)

GENERATING PASSAGE EMBEDDINGS


Batches: 100%|██████████| 80/80 [19:25<00:00, 14.57s/it] 


Embedding Summary
----------------------------------------
Passages           : 1,266
Embedding Shape    : (1266, 768)
Embedding Dimension: 768

Sample Passage


,Passage_ID,Passage_Text
0,1:1-4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين....



First 10 Values of First Embedding
[ 0.0072  0.0298 -0.0206  0.0099  0.0339 -0.0038 -0.0084 -0.0284  0.0276
  0.0467]


## Generate and Cache Query Embeddings

Each question in the QuranQA query collection is converted into a dense semantic vector using the pre-trained **multilingual-e5-base** embedding model.

Following the recommended E5 input format, every query is prefixed with `"query: "` before encoding. The resulting embeddings are normalized to unit length to enable efficient cosine similarity computation during retrieval.

To improve computational efficiency, previously generated query embeddings are automatically loaded if they already exist; otherwise, they are generated and stored for future use.

The following artifacts are created:

- **Query_Embeddings.npy** — Dense vector representations of all questions.
- **Query_Metadata.csv** — Mapping between each embedding and its corresponding question.

In [16]:
# ============================================================
# Generate and Cache Query Embeddings
# ============================================================

QUERY_EMBEDDINGS_FILE = (
    DENSE_RETRIEVAL_DIR /
    "Query_Embeddings.npy"
)

QUERY_METADATA_FILE = (
    DENSE_RETRIEVAL_DIR /
    "Query_Metadata.csv"
)

# ------------------------------------------------------------
# Load Existing Query Embeddings
# ------------------------------------------------------------

if QUERY_EMBEDDINGS_FILE.exists():

    print("=" * 70)
    print("LOADING EXISTING QUERY EMBEDDINGS")
    print("=" * 70)

    query_embeddings = np.load(QUERY_EMBEDDINGS_FILE)

    query_metadata = pd.read_csv(QUERY_METADATA_FILE)

# ------------------------------------------------------------
# Otherwise Generate Query Embeddings
# ------------------------------------------------------------

else:

    print("=" * 70)
    print("GENERATING QUERY EMBEDDINGS")
    print("=" * 70)

    # Prepare E5 Input Format

    query_texts = [
        "query: " + text
        for text in queries_df["Question"]
    ]

    # Generate Embeddings

    query_embeddings = model.encode(
        query_texts,
        batch_size=16,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Save Embeddings

    np.save(
        QUERY_EMBEDDINGS_FILE,
        query_embeddings
    )

    # Save Metadata

    query_metadata = queries_df[
        [
            "Question_ID",
            "Question"
        ]
    ].copy()

    query_metadata.to_csv(
        QUERY_METADATA_FILE,
        index=False,
        encoding="utf-8-sig"
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nEmbedding Summary")
print("-" * 40)

print(f"Queries             : {len(query_metadata):,}")
print(f"Embedding Shape     : {query_embeddings.shape}")
print(f"Embedding Dimension : {query_embeddings.shape[1]}")

print("\nSample Query")

display(
    query_metadata.head(1)
)

print("\nFirst 10 Values of First Query Embedding")

print(
    np.round(
        query_embeddings[0][:10],
        4
    )
)

GENERATING QUERY EMBEDDINGS


Batches: 100%|██████████| 16/16 [00:39<00:00,  2.48s/it]


Embedding Summary
----------------------------------------
Queries             : 251
Embedding Shape     : (251, 768)
Embedding Dimension : 768

Sample Query


,Question_ID,Question
0,101,من هم قوم شعيب؟



First 10 Values of First Query Embedding
[ 0.0221  0.0436 -0.0191  0.0262  0.0534 -0.001  -0.024  -0.0404  0.0222
 -0.0311]


## Compute Query–Passage Semantic Similarity

The semantic similarity between every query and every Qur'anic passage is computed using cosine similarity.

Because both the query and passage embeddings have been normalized to unit length during encoding, cosine similarity is equivalent to the dot product between the corresponding vectors. Therefore, matrix multiplication is used to efficiently compute the complete similarity matrix.

The resulting matrix contains one similarity score for every possible query–passage pair and serves as the basis for ranking passages in the subsequent retrieval stage.

In [17]:
# ============================================================
# Compute Query–Passage Similarity Matrix
# ============================================================

print("=" * 70)
print("COMPUTING QUERY–PASSAGE SIMILARITY")
print("=" * 70)

# ------------------------------------------------------------
# Compute Cosine Similarity
# (Embeddings are already normalized)
# ------------------------------------------------------------

similarity_matrix = np.matmul(
    query_embeddings,
    passage_embeddings.T
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nSimilarity Matrix Summary")
print("-" * 40)

print(f"Matrix Shape       : {similarity_matrix.shape}")
print(f"Total Comparisons  : {similarity_matrix.size:,}")

print(f"\nMinimum Similarity : {similarity_matrix.min():.4f}")
print(f"Maximum Similarity : {similarity_matrix.max():.4f}")
print(f"Average Similarity : {similarity_matrix.mean():.4f}")

print("\nSimilarity Scores for the First Query")
print("-" * 40)

print(
    np.round(
        similarity_matrix[0][:10],
        4
    )
)

COMPUTING QUERY–PASSAGE SIMILARITY

Similarity Matrix Summary
----------------------------------------
Matrix Shape       : (251, 1266)
Total Comparisons  : 317,766

Minimum Similarity : 0.6753
Maximum Similarity : 0.8785
Average Similarity : 0.7676

Similarity Scores for the First Query
----------------------------------------
[0.7164 0.7374 0.7459 0.7305 0.7465 0.7249 0.764  0.7446 0.7432 0.7312]


## Compute Dense Retrieval Scores

The semantic similarity matrix is converted into a tabular retrieval format by pairing every query with every Qur'anic passage.

For each query–passage pair, the corresponding cosine similarity score is recorded together with the query identifier, query text, and passage identifier.

The resulting retrieval table contains one record for every possible query–passage combination and forms the basis for the ranking stage in the next notebook cell.

In [18]:
# ============================================================
# Build Dense Retrieval Results
# ============================================================

print("=" * 70)
print("BUILDING DENSE RETRIEVAL RESULTS")
print("=" * 70)

retrieval_results = []

# ------------------------------------------------------------
# Build Query–Passage Similarity Table
# ------------------------------------------------------------

for query_idx, query_row in queries_df.iterrows():

    similarities = similarity_matrix[query_idx]

    for passage_idx, similarity in enumerate(similarities):

        retrieval_results.append({

            "Question_ID": query_row["Question_ID"],
            "Question": query_row["Question"],
            "Passage_ID": passages_df.iloc[passage_idx]["Passage_ID"],
            "Similarity_Score": float(similarity)

        })

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

dense_df = pd.DataFrame(retrieval_results)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nDense Retrieval Completed")
print("-" * 40)

print(f"Query–Passage Pairs : {len(dense_df):,}")
print(f"Questions Processed : {dense_df['Question_ID'].nunique():,}")
print(f"Passages Compared   : {passages_df.shape[0]:,}")

print("\nSimilarity Statistics")

print(f"Minimum Score : {dense_df['Similarity_Score'].min():.4f}")
print(f"Maximum Score : {dense_df['Similarity_Score'].max():.4f}")
print(f"Average Score : {dense_df['Similarity_Score'].mean():.4f}")

print("\nSample Retrieval Scores")

display(
    dense_df.head(10)
)

BUILDING DENSE RETRIEVAL RESULTS

Dense Retrieval Completed
----------------------------------------
Query–Passage Pairs : 317,766
Questions Processed : 251
Passages Compared   : 1,266

Similarity Statistics
Minimum Score : 0.6753
Maximum Score : 0.8785
Average Score : 0.7676

Sample Retrieval Scores


,Question_ID,Question,Passage_ID,Similarity_Score
0,101,من هم قوم شعيب؟,1:1-4,0.716368
1,101,من هم قوم شعيب؟,1:5-6,0.737393
2,101,من هم قوم شعيب؟,1:7-7,0.745943
3,101,من هم قوم شعيب؟,2:1-2,0.730502
4,101,من هم قوم شعيب؟,2:3-5,0.746493
5,101,من هم قوم شعيب؟,2:6-7,0.724943
6,101,من هم قوم شعيب؟,2:8-16,0.764028
7,101,من هم قوم شعيب؟,2:17-20,0.744590
8,101,من هم قوم شعيب؟,2:21-22,0.743161
9,101,من هم قوم شعيب؟,2:23-24,0.731188


## Rank Retrieved Passages

The semantic similarity scores are sorted in descending order for each query to produce the final ranked retrieval list.

Each Qur'anic passage is assigned a retrieval rank based on its cosine similarity score, where **Rank = 1** corresponds to the most semantically similar passage for a given question.

The ranked results will later be used to select the Top-K retrieved passages and evaluate the Dense Retrieval baseline using standard information retrieval metrics.

In [19]:
# ============================================================
# Rank Dense Retrieval Results
# ============================================================

print("=" * 70)
print("RANKING DENSE RETRIEVAL RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Sort by Similarity Score
# ------------------------------------------------------------

dense_df = dense_df.sort_values(
    by=["Question_ID", "Similarity_Score"],
    ascending=[True, False]
).reset_index(drop=True)

# ------------------------------------------------------------
# Assign Rank
# ------------------------------------------------------------

dense_df["Rank"] = (
    dense_df
    .groupby("Question_ID")
    .cumcount()
    + 1
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nRanking Completed")
print("-" * 40)

print(f"Total Ranked Pairs : {len(dense_df):,}")
print(f"Questions Ranked   : {dense_df['Question_ID'].nunique():,}")

print("\nTop-ranked passage for the first five questions")

display(
    dense_df
    .groupby("Question_ID")
    .head(1)
    .head(5)
)

print("\nRank Distribution")

print(
    dense_df["Rank"].describe()
)

RANKING DENSE RETRIEVAL RESULTS

Ranking Completed
----------------------------------------
Total Ranked Pairs : 317,766
Questions Ranked   : 251

Top-ranked passage for the first five questions


,Question_ID,Question,Passage_ID,Similarity_Score,Rank
0,101,من هم قوم شعيب؟,11:89-95,0.831902,1
1266,102,من هم قوم موسى؟,7:159-162,0.840296,1
2532,103,من بنى الكعبة؟,5:96-100,0.791290,1
3798,105,من هو النبي المعروف بالصبر؟,103:1-3,0.800197,1
5064,106,من كفل السيدة مريم؟,3:42-44,0.808361,1



Rank Distribution
count    317766.000000
mean        633.500000
std         365.463181
min           1.000000
25%         317.000000
50%         633.500000
75%         950.000000
max        1266.000000
Name: Rank, dtype: float64


## Select the Top-100 Retrieved Passages

To facilitate comparative evaluation, only the **Top-100** highest-ranked passages are retained for each question.

The resulting retrieval set represents the final output of the Dense Retrieval baseline and will be used in the evaluation notebook to compare retrieval performance against the BM25 baseline and the proposed RALSR framework.

Restricting the retrieval list to the Top-100 passages significantly reduces storage requirements while preserving all passages required for standard information retrieval evaluation metrics.

In [20]:
# ============================================================
# Keep Only Top-100 Ranked Passages
# ============================================================

TOP_K = 100

print("=" * 70)
print(f"SELECTING TOP-{TOP_K} RETRIEVED PASSAGES")
print("=" * 70)

# ------------------------------------------------------------
# Keep Only Top-K Passages per Question
# ------------------------------------------------------------

dense_topk_df = (
    dense_df
    .groupby("Question_ID", group_keys=False)
    .head(TOP_K)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nTop-K Retrieval Summary")
print("-" * 40)

print(f"Questions              : {dense_topk_df['Question_ID'].nunique():,}")
print(f"Top-K per Question     : {TOP_K}")
print(f"Total Retrieved Pairs  : {len(dense_topk_df):,}")

print("\nTop-5 Retrieved Passages for the First Question")

display(
    dense_topk_df
    .groupby("Question_ID")
    .head(5)
    .head(5)
)

SELECTING TOP-100 RETRIEVED PASSAGES

Top-K Retrieval Summary
----------------------------------------
Questions              : 251
Top-K per Question     : 100
Total Retrieved Pairs  : 25,100

Top-5 Retrieved Passages for the First Question


,Question_ID,Question,Passage_ID,Similarity_Score,Rank
0,101,من هم قوم شعيب؟,11:89-95,0.831902,1
1,101,من هم قوم شعيب؟,11:84-88,0.819672,2
2,101,من هم قوم شعيب؟,7:85-93,0.813377,3
3,101,من هم قوم شعيب؟,18:9-16,0.778903,4
4,101,من هم قوم شعيب؟,4:36-40,0.778333,5


## Export Dense Retrieval Results

The final Dense Retrieval results are exported in two formats:

1. **Complete Ranked Retrieval Results** — Contains the full ranking of all query–passage pairs and preserves the complete retrieval output for reproducibility and future analyses.

2. **Top-100 Retrieval Results** — Contains only the highest-ranked 100 passages for each question and serves as the primary input for the comparative evaluation experiments.

Both files are stored in the **Dense_Retrieval** experiment directory.

In [21]:
# ============================================================
# Export Dense Retrieval Results
# ============================================================

ALL_RESULTS_FILE = (
    DENSE_RETRIEVAL_DIR /
    "Dense_Retrieval_All_Ranked.csv"
)

TOP100_RESULTS_FILE = (
    DENSE_RETRIEVAL_DIR /
    "Dense_Retrieval_Top100.csv"
)

print("=" * 70)
print("EXPORTING DENSE RETRIEVAL RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Export Complete Ranked Results
# ------------------------------------------------------------

dense_df.to_csv(
    ALL_RESULTS_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Export Top-100 Results
# ------------------------------------------------------------

dense_topk_df.to_csv(
    TOP100_RESULTS_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Export Summary
# ------------------------------------------------------------

print("\nExport Completed Successfully")
print("-" * 40)

print(f"Complete Ranked Results : {ALL_RESULTS_FILE.name}")
print(f"Top-100 Results         : {TOP100_RESULTS_FILE.name}")

print("\nFile Statistics")
print("-" * 40)

print(f"Complete Ranked Rows : {len(dense_df):,}")
print(f"Top-100 Rows         : {len(dense_topk_df):,}")

print("\nFiles Saved To")

print(DENSE_RETRIEVAL_DIR)

EXPORTING DENSE RETRIEVAL RESULTS

Export Completed Successfully
----------------------------------------
Complete Ranked Results : Dense_Retrieval_All_Ranked.csv
Top-100 Results         : Dense_Retrieval_Top100.csv

File Statistics
----------------------------------------
Complete Ranked Rows : 317,766
Top-100 Rows         : 25,100

Files Saved To
${LOCAL_USER_HOME}\Desktop\Alatoo university CS\NLP Thesis\05_Experiments\Dense_Retrieval


# Notebook Summary

## Objective

This notebook implemented the **Dense Retrieval baseline** for the QuranQA retrieval framework using the pre-trained **multilingual-e5-base** transformer model. Unlike BM25, which relies on lexical matching, Dense Retrieval represents both questions and Qur'anic passages as dense semantic embeddings and retrieves passages based on cosine similarity in a shared semantic embedding space.

## Workflow Completed

The following stages were successfully completed:

- Loaded the preprocessed QuranQA question collection.
- Loaded the processed Qur'anic passage collection.
- Loaded the locally stored **multilingual-e5-base** embedding model.
- Verified the embedding model using a sample query.
- Generated dense embeddings for all **1,266 Qur'anic passages**.
- Cached passage embeddings for efficient reuse.
- Generated dense embeddings for all **251 questions**.
- Cached query embeddings for efficient reuse.
- Computed the semantic similarity matrix between every query and every passage.
- Constructed the complete Dense Retrieval result table.
- Ranked all retrieved passages according to cosine similarity.
- Selected the **Top-100** passages for each query.
- Exported both the complete ranked retrieval results and the Top-100 retrieval results.

## Output Files

The following files were generated in the **Dense_Retrieval** experiment directory:

- `Passage_Embeddings.npy`
- `Passage_Metadata.csv`
- `Query_Embeddings.npy`
- `Query_Metadata.csv`
- `Dense_Retrieval_All_Ranked.csv`
- `Dense_Retrieval_Top100.csv`

## Experimental Summary

| Item | Value |
|------|------:|
| Questions | 251 |
| Qur'anic Passages | 1,266 |
| Embedding Model | multilingual-e5-base |
| Embedding Dimension | 768 |
| Total Query–Passage Comparisons | 317,766 |
| Retrieval Depth | Top-100 |

## Next Notebook

The next notebook (**Notebook 10C – Comparative Evaluation**) will evaluate and compare the retrieval effectiveness of:

- Baseline 1 — BM25
- Baseline 2 — Dense Retrieval
- Proposed Method — RALSR

using standard Information Retrieval evaluation metrics, including **Recall@K**, **MRR (Mean Reciprocal Rank)**, **MAP (Mean Average Precision)**, and **nDCG (Normalized Discounted Cumulative Gain)**.